In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import kagglehub
import os
import shutil

# Download latest version
path = kagglehub.dataset_download("andradaolteanu/gtzan-dataset-music-genre-classification")

print("Dataset downloaded to:", path)
print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)

In [ ]:
import os
import shutil
import random

# 1. Define paths
source_dir = '/kaggle/input/gtzan-dataset-music-genre-classification/Data/genres_original'
target_root = 'gtzan_splits'
splits = ['train', 'val', 'test']

# 2. Create directory structure
genres = [d for d in os.listdir(source_dir) if os.path.isdir(os.path.join(source_dir, d))]
for split in splits:
    for genre in genres:
        os.makedirs(os.path.join(target_root, split, genre), exist_ok=True)

# 3-5. Iterate, Split, and Copy
random.seed(42)  # For reproducibility
stats = {split: 0 for split in splits}

for genre in genres:
    genre_path = os.path.join(source_dir, genre)
    files = [f for f in os.listdir(genre_path) if f.endswith('.wav')]
    random.shuffle(files)

    n = len(files)
    train_idx = int(n * 0.8)
    val_idx = int(n * 0.9)

    file_splits = {
        'train': files[:train_idx],
        'val': files[train_idx:val_idx],
        'test': files[val_idx:]
    }

    for split, split_files in file_splits.items():
        for f in split_files:
            src = os.path.join(genre_path, f)
            dst = os.path.join(target_root, split, genre, f)
            shutil.copy(src, dst)
            stats[split] += 1

# 6. Print Verification
print("Data split complete.")
for split, count in stats.items():
    print(f"{split.capitalize()} set: {count} files")

In [ ]:
import torch
import torchaudio
import os
from torch.utils.data import Dataset
import torch.nn.functional as F

class GTZANDataset(Dataset):
    def __init__(self, root_dir, split, sample_rate=22050, duration=15, augment=False):
        self.root_dir = os.path.join(root_dir, split)
        self.sample_rate = sample_rate
        self.duration = duration
        self.n_samples = sample_rate * duration
        self.augment = augment
        self.file_list = []

        # Map genres to integers
        genres = sorted([d for d in os.listdir(self.root_dir) if os.path.isdir(os.path.join(self.root_dir, d))])
        self.label_to_idx = {genre: i for i, genre in enumerate(genres)}

        for genre in genres:
            genre_dir = os.path.join(self.root_dir, genre)
            for f in os.listdir(genre_dir):
                if f.endswith('.wav'):
                    full_path = os.path.join(genre_dir, f)
                    for segment_idx in [0, 1]:
                        # Original segment
                        self.file_list.append((full_path, self.label_to_idx[genre], segment_idx, 'none'))
                        if self.augment:
                            # Augmented versions
                            self.file_list.append((full_path, self.label_to_idx[genre], segment_idx, 'pitch_up'))
                            self.file_list.append((full_path, self.label_to_idx[genre], segment_idx, 'pitch_down'))
                            self.file_list.append((full_path, self.label_to_idx[genre], segment_idx, 'noise'))

        # Initialize transforms
        self.pitch_shift_up = torchaudio.transforms.PitchShift(sample_rate, n_steps=1)
        self.pitch_shift_down = torchaudio.transforms.PitchShift(sample_rate, n_steps=-1)

    def add_white_noise(self, waveform, noise_level=0.005):
        noise = torch.randn_like(waveform) * noise_level
        return waveform + noise

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        path, label, segment_idx, aug_type = self.file_list[idx]
        offset = segment_idx * self.n_samples
        waveform, sr = torchaudio.load(path, frame_offset=offset, num_frames=self.n_samples)

        if sr != self.sample_rate:
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=self.sample_rate)
            waveform = resampler(waveform)

        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        if waveform.shape[1] < self.n_samples:
            waveform = F.pad(waveform, (0, self.n_samples - waveform.shape[1]))
        else:
            waveform = waveform[:, :self.n_samples]

        # Apply Augmentations
        if aug_type == 'pitch_up':
            waveform = self.pitch_shift_up(waveform)
        elif aug_type == 'pitch_down':
            waveform = self.pitch_shift_down(waveform)
        elif aug_type == 'noise':
            waveform = self.add_white_noise(waveform)

        if waveform.abs().max() > 0:
            waveform = waveform / waveform.abs().max()

        return waveform.squeeze(0), torch.tensor(label)

In [ ]:
from torch.utils.data import DataLoader

# Initialize Datasets (augmentation enabled for training)
train_dataset = GTZANDataset(root_dir='gtzan_split', split='train', augment=True)
val_dataset = GTZANDataset(root_dir='gtzan_split', split='val', augment=False)
test_dataset = GTZANDataset(root_dir='gtzan_split', split='test', augment=False)

# Initialize DataLoaders
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Verification
audios, labels = next(iter(train_loader))
print(f'Train Dataset Size (Augmented): {len(train_dataset)}')
print(f'Audio Shape: {audios.shape}')
print('DataLoaders updated with pitch-shift and Gaussian noise augmentation.')